# 02 — Silver Transformation Layer
## Well Production Forecasting & Performance Analytics System

**Medallion Architecture — Silver Layer**

Transforms Bronze raw data into analytics-ready Silver tables:
- Data quality checks: null counts before/after cleaning
- Type casting: strings → DateType, DoubleType, IntegerType
- Column standardization: snake_case naming
- Deduplication on business keys
- Derived columns: `production_year`, `production_quarter`
- Join: production records enriched with well metadata
- Partitioned by `production_year`

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .master('local[*]')
    .appName('WellAnalytics-Silver')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.databricks.delta.schema.autoMerge.enabled', 'true')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '4g')
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready')

## Step 1 — Data Quality Report on Bronze Production

In [ ]:
from pyspark.sql import functions as F
from src.spark_session import bronze_path

prod_raw = spark.read.format('delta').load(bronze_path('bronze_well_production'))
total_rows = prod_raw.count()

print(f'Bronze production rows: {total_rows:,}')
print('\nNull counts per column:')
null_counts = prod_raw.select([
    F.count(F.when(F.col(c).isNull() | (F.col(c) == ''), c)).alias(c)
    for c in prod_raw.columns
])
null_counts.show(vertical=True)

## Step 2 — Run Silver Transformations

In [ ]:
from src.silver_transform import run_silver

counts = run_silver(spark)
print('\nSilver table row counts:')
for t, n in counts.items():
    print(f'  {t:<30s} {n:>8,}')

## Step 3 — Validate Silver Schemas

In [ ]:
from src.spark_session import silver_path

for table in ['silver_wells', 'silver_flaring', 'silver_production']:
    df = spark.read.format('delta').load(silver_path(table))
    print(f'\n── {table} ({df.count():,} rows) ──')
    df.printSchema()
    df.show(3)

## Step 4 — Ad-hoc SQL Exploration (Databricks-style)

In [ ]:
from src.spark_session import silver_path

# Register as temp views for SQL exploration
spark.read.format('delta').load(silver_path('silver_production')).createOrReplaceTempView('silver_production')
spark.read.format('delta').load(silver_path('silver_flaring')).createOrReplaceTempView('silver_flaring')
spark.read.format('delta').load(silver_path('silver_wells')).createOrReplaceTempView('silver_wells')

spark.sql("""
SELECT
    operator,
    production_year,
    production_quarter,
    oil_and_gas_group,
    ROUND(SUM(production), 2) AS total_production,
    COUNT(DISTINCT api_number) AS active_wells
FROM silver_production
GROUP BY operator, production_year, production_quarter, oil_and_gas_group
ORDER BY operator, production_year, production_quarter
""").show(20)

## Step 5 — Data Quality Report Post-Silver

In [ ]:
silver_prod = spark.read.format('delta').load(silver_path('silver_production'))
bronze_count = spark.read.format('delta').load(bronze_path('bronze_well_production')).count()
silver_count = silver_prod.count()

print('Data Quality Summary')
print('=' * 45)
print(f'  Bronze rows       : {bronze_count:>10,}')
print(f'  Silver rows       : {silver_count:>10,}')
print(f'  Retention rate    : {silver_count/bronze_count*100:>9.1f}%')
print(f'  Unique wells      : {silver_prod.select("api_number").distinct().count():>10,}')
print(f'  Unique operators  : {silver_prod.select("operator").distinct().count():>10,}')
print(f'  Date range        : {silver_prod.agg({"production_month": "min"}).collect()[0][0]} '
      f'to {silver_prod.agg({"production_month": "max"}).collect()[0][0]}')
print(f'  Null productions  : {silver_prod.filter("production IS NULL").count():>10,}')